# Unified Old Photo Restoration + Exemplar Colorization Pipeline

This notebook combines:
1. **Old Photo Restoration** - Remove scratches, noise, and artifacts
2. **Deep Exemplar-based Colorization** - Colorize using reference images

**Pipeline**: Damaged B&W → Restoration → Exemplar Colorization → Final Color Image

## Setup Instructions
1. Run all cells in order
2. Training takes ~6-7 hours on Colab GPU
3. Download trained checkpoints from `/content/checkpoints/`
4. Use downloaded checkpoints with app.py locally

## 1. Environment Setup

In [ ]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab")
    !nvidia-smi
else:
    print("Running locally")

In [ ]:
# Install dependencies
!pip install torch torchvision scikit-image scikit-learn opencv-python dominate einops tensorboardX scipy Pillow numpy tqdm

In [ ]:
# Clone or mount project
if IN_COLAB:
    # Option 1: Clone from GitHub
    !git clone https://github.com/YOUR_USERNAME/bw2colour.git
    %cd bw2colour
    
    # Option 2: Mount Google Drive (uncomment if you uploaded project to Drive)
    # from google.colab import drive
    # drive.mount('/content/drive')
    # %cd /content/drive/MyDrive/bw2colour

## 2. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models

import numpy as np
import cv2
from PIL import Image
import os
import matplotlib.pyplot as plt
from skimage import color
from tqdm import tqdm
import zipfile
import urllib.request

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 3. Dataset Preparation

In [ ]:
# Create directory structure
os.makedirs('datasets/old_photos', exist_ok=True)
os.makedirs('datasets/color_images', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("Directory structure created!")

In [ ]:
# Download sample datasets (you can replace with your own data)
# For restoration training, we need:
# 1. Old/degraded photos
# 2. Clean reference photos
# 3. Color reference images for colorization

# Example: Download from a public dataset or use your own
# Uncomment and modify for your dataset source:

# !wget -O datasets.zip YOUR_DATASET_URL
# !unzip datasets.zip -d datasets/

print("Dataset preparation:")
print("- Place old/degraded photos in: datasets/old_photos/")
print("- Place color reference images in: datasets/color_images/")
print("- Or upload your own datasets")

## 4. Define Dataset Classes

In [ ]:
class RestorationDataset(Dataset):
    """Dataset for training old photo restoration"""
    
    def __init__(self, root_dir, transform=None, add_synthetic_degradation=True):
        self.root_dir = root_dir
        self.transform = transform
        self.add_synthetic_degradation = add_synthetic_degradation
        
        # Get all image files
        self.image_files = [f for f in os.listdir(root_dir) 
                           if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    def add_degradation(self, img):
        """Add synthetic degradation to simulate old photos"""
        img_np = np.array(img).astype(np.float32) / 255.0
        
        # Add noise
        noise = np.random.normal(0, 0.1, img_np.shape)
        img_np = np.clip(img_np + noise, 0, 1)
        
        # Add scratches (random lines)
        if np.random.rand() > 0.5:
            h, w = img_np.shape[:2]
            for _ in range(np.random.randint(3, 10)):
                pt1 = (np.random.randint(0, w), np.random.randint(0, h))
                pt2 = (np.random.randint(0, w), np.random.randint(0, h))
                cv2.line(img_np, pt1, pt2, (1, 1, 1), np.random.randint(1, 3))
        
        # Add blur
        if np.random.rand() > 0.5:
            kernel_size = np.random.choice([3, 5])
            img_np = cv2.GaussianBlur(img_np, (kernel_size, kernel_size), 0)
        
        return (img_np * 255).astype(np.uint8)
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        clean_img = Image.open(img_path).convert('RGB')
        
        # Create degraded version
        if self.add_synthetic_degradation:
            degraded_img = Image.fromarray(self.add_degradation(clean_img))
        else:
            degraded_img = clean_img
        
        if self.transform:
            clean_img = self.transform(clean_img)
            degraded_img = self.transform(degraded_img)
        
        return degraded_img, clean_img


class ColorizationDataset(Dataset):
    """Dataset for training exemplar-based colorization"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) 
                           if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    def rgb2lab(self, img):
        """Convert RGB to LAB color space"""
        img_lab = color.rgb2lab(img)
        return img_lab
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        # Load color image
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        color_img = Image.open(img_path).convert('RGB')
        
        if self.transform:
            color_img = self.transform(color_img)
        
        # Convert to numpy for LAB conversion
        color_np = np.array(color_img).astype(np.float32) / 255.0
        lab_img = self.rgb2lab(color_np)
        
        # Split L and AB channels
        L = lab_img[:, :, 0:1].transpose((2, 0, 1))  # (1, H, W)
        AB = lab_img[:, :, 1:3].transpose((2, 0, 1))  # (2, H, W)
        
        # Normalize
        L = (L - 50.0) / 100.0  # Normalize L channel
        AB = AB / 110.0  # Normalize AB channels
        
        # For exemplar colorization, use random reference from same image
        # In practice, you'd load a different reference image
        ref_idx = np.random.randint(0, len(self.image_files))
        ref_path = os.path.join(self.root_dir, self.image_files[ref_idx])
        ref_img = Image.open(ref_path).convert('RGB')
        
        if self.transform:
            ref_img = self.transform(ref_img)
        
        ref_np = np.array(ref_img).astype(np.float32) / 255.0
        
        return {
            'L': torch.FloatTensor(L),
            'AB': torch.FloatTensor(AB),
            'reference': torch.FloatTensor(ref_np.transpose((2, 0, 1)))
        }

print("Dataset classes defined!")

## 5. Define Model Architectures

In [ ]:
# Import similarity subnet
import sys
sys.path.append('..')
from similarity_pytorch import VGG19Features, SimilaritySubnet

In [ ]:
# ===== Restoration Model (U-Net based) =====
class RestorationUNet(nn.Module):
    """U-Net architecture for old photo restoration"""
    
    def __init__(self, in_channels=3, out_channels=3):
        super(RestorationUNet, self).__init__()
        
        # Encoder
        self.enc1 = self.conv_block(in_channels, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.enc4 = self.conv_block(256, 512)
        
        # Bottleneck
        self.bottleneck = self.conv_block(512, 1024)
        
        # Decoder
        self.dec4 = self.upconv_block(1024, 512)
        self.dec3 = self.upconv_block(512, 256)
        self.dec2 = self.upconv_block(256, 128)
        self.dec1 = self.upconv_block(128, 64)
        
        # Final output
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)
        
        self.pool = nn.MaxPool2d(2, 2)
    
    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def upconv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e4))
        
        # Decoder with skip connections
        d4 = self.dec4(b)
        d4 = torch.cat([d4, e4], dim=1) if d4.shape[2:] == e4.shape[2:] else d4
        d3 = self.dec3(d4)
        d3 = torch.cat([d3, e3], dim=1) if d3.shape[2:] == e3.shape[2:] else d3
        d2 = self.dec2(d3)
        d2 = torch.cat([d2, e2], dim=1) if d2.shape[2:] == e2.shape[2:] else d2
        d1 = self.dec1(d2)
        
        out = self.final(d1)
        return torch.tanh(out)  # Output in [-1, 1]


# ===== Colorization Model (from existing implementation) =====
class ExampleColorNet(nn.Module):
    """Exemplar-based colorization network"""
    
    def __init__(self, ic):
        super(ExampleColorNet, self).__init__()
        self.conv1_1 = nn.Sequential(nn.Conv2d(ic, 32, 3, 1, 1), nn.ReLU(),
                                     nn.Conv2d(32, 64, 3, 1, 1))
        self.conv1_2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv1_2norm = nn.BatchNorm2d(64, affine=False)
        self.conv1_2norm_ss = nn.Conv2d(64, 64, 1, 2, bias=False, groups=64)
        self.conv2_1 = nn.Conv2d(64, 128, 3, 1, 1)
        self.conv2_2 = nn.Conv2d(128, 128, 3, 1, 1)
        self.conv2_2norm = nn.BatchNorm2d(128, affine=False)
        self.conv2_2norm_ss = nn.Conv2d(128, 128, 1, 2, bias=False, groups=128)
        self.conv3_1 = nn.Conv2d(128, 256, 3, 1, 1)
        self.conv3_2 = nn.Conv2d(256, 256, 3, 1, 1)
        self.conv3_3 = nn.Conv2d(256, 256, 3, 1, 1)
        self.conv3_3norm = nn.BatchNorm2d(256, affine=False)
        self.conv3_3norm_ss = nn.Conv2d(256, 256, 1, 2, bias=False, groups=256)
        self.conv4_1 = nn.Conv2d(256, 512, 3, 1, 1)
        self.conv4_2 = nn.Conv2d(512, 512, 3, 1, 1)
        self.conv4_3 = nn.Conv2d(512, 512, 3, 1, 1)
        self.conv4_3norm = nn.BatchNorm2d(512, affine=False)
        self.conv5_1 = nn.Conv2d(512, 512, 3, 1, 2, 2)
        self.conv5_2 = nn.Conv2d(512, 512, 3, 1, 2, 2)
        self.conv5_3 = nn.Conv2d(512, 512, 3, 1, 2, 2)
        self.conv5_3norm = nn.BatchNorm2d(512, affine=False)
        self.conv6_1 = nn.Conv2d(512, 512, 3, 1, 2, 2)
        self.conv6_2 = nn.Conv2d(512, 512, 3, 1, 2, 2)
        self.conv6_3 = nn.Conv2d(512, 512, 3, 1, 2, 2)
        self.conv6_3norm = nn.BatchNorm2d(512, affine=False)
        self.conv7_1 = nn.Conv2d(512, 512, 3, 1, 1)
        self.conv7_2 = nn.Conv2d(512, 512, 3, 1, 1)
        self.conv7_3 = nn.Conv2d(512, 512, 3, 1, 1)
        self.conv7_3norm = nn.BatchNorm2d(512, affine=False)
        self.conv8_1 = nn.ConvTranspose2d(512, 256, 4, 2, 1)
        self.conv3_3_short = nn.Conv2d(256, 256, 3, 1, 1)
        self.conv8_2 = nn.Conv2d(256, 256, 3, 1, 1)
        self.conv8_3 = nn.Conv2d(256, 256, 3, 1, 1)
        self.conv8_3norm = nn.BatchNorm2d(256, affine=False)
        self.conv9_1 = nn.ConvTranspose2d(256, 128, 4, 2, 1)
        self.conv2_2_short = nn.Conv2d(128, 128, 3, 1, 1)
        self.conv9_2 = nn.Conv2d(128, 128, 3, 1, 1)
        self.conv9_2norm = nn.BatchNorm2d(128, affine=False)
        self.conv10_1 = nn.ConvTranspose2d(128, 128, 4, 2, 1)
        self.conv1_2_short = nn.Conv2d(64, 128, 3, 1, 1)
        self.conv10_2 = nn.Conv2d(128, 128, 3, 1, 1)
        self.conv10_ab = nn.Conv2d(128, 2, 1, 1)
        self.leaky_relu = nn.LeakyReLU(0.2, True)

    def forward(self, x):
        conv1_1 = torch.relu(self.conv1_1(x))
        conv1_2 = torch.relu(self.conv1_2(conv1_1))
        conv1_2norm = self.conv1_2norm(conv1_2)
        conv1_2norm_ss = self.conv1_2norm_ss(conv1_2norm)
        conv2_1 = torch.relu(self.conv2_1(conv1_2norm_ss))
        conv2_2 = torch.relu(self.conv2_2(conv2_1))
        conv2_2norm = self.conv2_2norm(conv2_2)
        conv2_2norm_ss = self.conv2_2norm_ss(conv2_2norm)
        conv3_1 = torch.relu(self.conv3_1(conv2_2norm_ss))
        conv3_2 = torch.relu(self.conv3_2(conv3_1))
        conv3_3 = torch.relu(self.conv3_3(conv3_2))
        conv3_3norm = self.conv3_3norm(conv3_3)
        conv3_3norm_ss = self.conv3_3norm_ss(conv3_3norm)
        conv4_1 = torch.relu(self.conv4_1(conv3_3norm_ss))
        conv4_2 = torch.relu(self.conv4_2(conv4_1))
        conv4_3 = torch.relu(self.conv4_3(conv4_2))
        conv4_3norm = self.conv4_3norm(conv4_3)
        conv5_1 = torch.relu(self.conv5_1(conv4_3norm))
        conv5_2 = torch.relu(self.conv5_2(conv5_1))
        conv5_3 = torch.relu(self.conv5_3(conv5_2))
        conv5_3norm = self.conv5_3norm(conv5_3)
        conv6_1 = torch.relu(self.conv6_1(conv5_3norm))
        conv6_2 = torch.relu(self.conv6_2(conv6_1))
        conv6_3 = torch.relu(self.conv6_3(conv6_2))
        conv6_3norm = self.conv6_3norm(conv6_3)
        conv7_1 = torch.relu(self.conv7_1(conv6_3norm))
        conv7_2 = torch.relu(self.conv7_2(conv7_1))
        conv7_3 = torch.relu(self.conv7_3(conv7_2))
        conv7_3norm = self.conv7_3norm(conv7_3)
        conv8_1 = self.conv8_1(conv7_3norm)
        conv3_3_short = self.conv3_3_short(conv3_3norm)
        conv8_1_comb = torch.relu(conv8_1 + conv3_3_short)
        conv8_2 = torch.relu(self.conv8_2(conv8_1_comb))
        conv8_3 = torch.relu(self.conv8_3(conv8_2))
        conv8_3norm = self.conv8_3norm(conv8_3)
        conv9_1 = self.conv9_1(conv8_3norm)
        conv2_2_short = self.conv2_2_short(conv2_2norm)
        conv9_1_comb = torch.relu(conv9_1 + conv2_2_short)
        conv9_2 = torch.relu(self.conv9_2(conv9_1_comb))
        conv9_2norm = self.conv9_2norm(conv9_2)
        conv10_1 = self.conv10_1(conv9_2norm)
        conv1_2_short = self.conv1_2_short(conv1_2norm)
        conv10_1_comb = torch.relu(conv10_1 + conv1_2_short)
        conv10_2 = self.leaky_relu(self.conv10_2(conv10_1_comb))
        conv10_ab = self.conv10_ab(conv10_2)
        pred_ab = torch.tanh(conv10_ab) * 100
        return pred_ab

print("Model architectures defined!")

## 6. Training Functions

In [ ]:
def train_restoration_model(model, dataloader, num_epochs=50, lr=0.0002):
    """Train the restoration model"""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion_l1 = nn.L1Loss()
    criterion_mse = nn.MSELoss()
    
    # For perceptual loss
    vgg = models.vgg19(pretrained=True).features[:36].to(device)
    vgg.eval()
    for param in vgg.parameters():
        param.requires_grad = False
    
    train_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for degraded, clean in pbar:
            degraded = degraded.to(device)
            clean = clean.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            restored = model(degraded)
            
            # Compute losses
            loss_l1 = criterion_l1(restored, clean)
            loss_mse = criterion_mse(restored, clean)
            
            # Perceptual loss
            restored_feat = vgg(restored)
            clean_feat = vgg(clean)
            loss_perceptual = criterion_mse(restored_feat, clean_feat)
            
            # Total loss
            loss = loss_l1 + 0.1 * loss_mse + 0.01 * loss_perceptual
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_loss = epoch_loss / len(dataloader)
        train_losses.append(avg_loss)
        print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")
        
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, f'checkpoints/restoration_epoch_{epoch+1}.pth')
            print(f"Checkpoint saved: restoration_epoch_{epoch+1}.pth")
    
    return train_losses


def train_colorization_model(model, dataloader, similarity_subnet, num_epochs=50, lr=0.0002):
    """Train the colorization model"""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            L = batch['L'].to(device)
            AB_gt = batch['AB'].to(device)
            reference = batch['reference'].to(device)
            
            optimizer.zero_grad()
            
            # Compute similarity maps (detached, not trained)
            with torch.no_grad():
                sim_maps = similarity_subnet.compute_similarity(
                    L, reference, layers=['conv3_1', 'conv4_1']
                )
            
            # Concatenate L channel with similarity information
            # For simplicity, we'll use L + reference
            input_tensor = torch.cat([L, reference], dim=1)
            
            # Forward pass
            AB_pred = model(input_tensor)
            
            # Compute loss
            loss = criterion(AB_pred / 100.0, AB_gt)
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_loss = epoch_loss / len(dataloader)
        train_losses.append(avg_loss)
        print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")
        
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, f'checkpoints/colorization_epoch_{epoch+1}.pth')
            print(f"Checkpoint saved: colorization_epoch_{epoch+1}.pth")
    
    return train_losses

print("Training functions defined!")

## 7. Train Restoration Model

In [ ]:
# Setup data transforms
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Create restoration dataset
restoration_dataset = RestorationDataset(
    'datasets/color_images',  # Use color images and add synthetic degradation
    transform=transform,
    add_synthetic_degradation=True
)

restoration_loader = DataLoader(
    restoration_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2
)

print(f"Restoration dataset size: {len(restoration_dataset)}")
print(f"Number of batches: {len(restoration_loader)}")

In [ ]:
# Initialize restoration model
restoration_model = RestorationUNet(in_channels=3, out_channels=3)

# Train (reduce epochs for faster training if needed)
print("Starting restoration model training...")
restoration_losses = train_restoration_model(
    restoration_model,
    restoration_loader,
    num_epochs=50,  # Adjust based on your time budget
    lr=0.0002
)

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(restoration_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Restoration Model Training Loss')
plt.grid(True)
plt.savefig('outputs/restoration_training_loss.png')
plt.show()

# Save final model
torch.save(restoration_model.state_dict(), 'checkpoints/restoration_final.pth')
print("Restoration model training complete!")

## 8. Train Colorization Model

In [ ]:
# Create colorization dataset
color_transform = transforms.Compose([
    transforms.Resize((256, 256)),
])

colorization_dataset = ColorizationDataset(
    'datasets/color_images',
    transform=color_transform
)

colorization_loader = DataLoader(
    colorization_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2
)

print(f"Colorization dataset size: {len(colorization_dataset)}")
print(f"Number of batches: {len(colorization_loader)}")

In [ ]:
# Initialize colorization model and similarity subnet
colorization_model = ExampleColorNet(ic=4)  # 1 (L) + 3 (reference RGB)
similarity_subnet = SimilaritySubnet(device=device)

# Train
print("Starting colorization model training...")
colorization_losses = train_colorization_model(
    colorization_model,
    colorization_loader,
    similarity_subnet,
    num_epochs=50,
    lr=0.0002
)

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(colorization_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Colorization Model Training Loss')
plt.grid(True)
plt.savefig('outputs/colorization_training_loss.png')
plt.show()

# Save final model
torch.save(colorization_model.state_dict(), 'checkpoints/colorization_final.pth')
print("Colorization model training complete!")

## 9. End-to-End Inference Pipeline

In [ ]:
def lab2rgb(L, AB):
    """Convert LAB to RGB"""
    # Denormalize
    L = L * 100.0 + 50.0
    AB = AB * 110.0
    
    # Concatenate
    lab = np.concatenate([L, AB], axis=2)
    
    # Convert to RGB
    rgb = color.lab2rgb(lab)
    rgb = (rgb * 255).astype(np.uint8)
    
    return rgb


def unified_pipeline_inference(old_photo_path, reference_path, restoration_model, colorization_model, similarity_subnet):
    """
    Complete pipeline: Old/Damaged B&W → Restoration → Colorization → Final Color Image
    """
    # Load models in eval mode
    restoration_model.eval()
    colorization_model.eval()
    
    # 1. Load old photo
    old_photo = Image.open(old_photo_path).convert('RGB')
    original_size = old_photo.size
    
    # 2. Restoration
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    old_tensor = transform(old_photo).unsqueeze(0).to(device)
    
    with torch.no_grad():
        restored_tensor = restoration_model(old_tensor)
    
    # Denormalize and convert to grayscale
    restored_tensor = (restored_tensor + 1) / 2  # [-1, 1] -> [0, 1]
    restored_np = restored_tensor.squeeze(0).cpu().numpy().transpose(1, 2, 0)
    restored_gray = color.rgb2gray(restored_np)
    
    # 3. Colorization
    # Load reference
    reference = Image.open(reference_path).convert('RGB')
    reference = reference.resize((256, 256))
    reference_np = np.array(reference).astype(np.float32) / 255.0
    reference_tensor = torch.from_numpy(reference_np.transpose(2, 0, 1)).unsqueeze(0).to(device)
    
    # Convert restored to L channel
    L = (restored_gray * 100.0 - 50.0) / 100.0  # Normalize
    L_tensor = torch.from_numpy(L).unsqueeze(0).unsqueeze(0).float().to(device)
    
    # Compute similarity
    with torch.no_grad():
        sim_maps = similarity_subnet.compute_similarity(
            L_tensor, reference_tensor, layers=['conv3_1', 'conv4_1']
        )
    
    # Colorize
    input_tensor = torch.cat([L_tensor, reference_tensor], dim=1)
    
    with torch.no_grad():
        AB_pred = colorization_model(input_tensor)
    
    # Convert to RGB
    L_out = L_tensor.squeeze().cpu().numpy()
    AB_out = AB_pred.squeeze().cpu().numpy() / 110.0  # Normalize
    
    L_out = L_out[np.newaxis, :, :].transpose(1, 2, 0)
    AB_out = AB_out.transpose(1, 2, 0)
    
    colorized = lab2rgb(L_out, AB_out)
    
    # 4. Resize back to original size
    colorized_final = Image.fromarray(colorized).resize(original_size, Image.LANCZOS)
    restored_final = Image.fromarray((restored_np * 255).astype(np.uint8)).resize(original_size, Image.LANCZOS)
    
    return {
        'original': old_photo,
        'restored': restored_final,
        'colorized': colorized_final,
        'reference': reference
    }

print("Inference pipeline defined!")

## 10. Test the Complete Pipeline

In [ ]:
# Load trained models
restoration_model_test = RestorationUNet()
restoration_model_test.load_state_dict(torch.load('checkpoints/restoration_final.pth'))
restoration_model_test = restoration_model_test.to(device)

colorization_model_test = ExampleColorNet(ic=4)
colorization_model_test.load_state_dict(torch.load('checkpoints/colorization_final.pth'))
colorization_model_test = colorization_model_test.to(device)

similarity_subnet_test = SimilaritySubnet(device=device)

print("Models loaded successfully!")

In [ ]:
# Test on sample images
# Replace with your actual test image paths
test_old_photo = 'restoration/test_images/old/test1.jpg'  # Update path
test_reference = 'datasets/color_images/ref1.jpg'  # Update path

if os.path.exists(test_old_photo) and os.path.exists(test_reference):
    results = unified_pipeline_inference(
        test_old_photo,
        test_reference,
        restoration_model_test,
        colorization_model_test,
        similarity_subnet_test
    )
    
    # Visualize results
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    axes[0].imshow(results['original'])
    axes[0].set_title('Original (Damaged)')
    axes[0].axis('off')
    
    axes[1].imshow(results['restored'])
    axes[1].set_title('Restored')
    axes[1].axis('off')
    
    axes[2].imshow(results['reference'])
    axes[2].set_title('Reference')
    axes[2].axis('off')
    
    axes[3].imshow(results['colorized'])
    axes[3].set_title('Final Colorized')
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.savefig('outputs/pipeline_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Save individual results
    results['colorized'].save('outputs/final_colorized.png')
    print("Results saved to outputs/")
else:
    print("Test images not found. Please update paths or add test images.")

## 11. Download Checkpoints (For Local Use)

In [ ]:
# Zip all checkpoints for easy download
if IN_COLAB:
    !zip -r checkpoints.zip checkpoints/
    print("Checkpoints zipped! Download checkpoints.zip from the Files panel.")
    print("Extract and place in your local project's checkpoints/ directory.")
else:
    print("Checkpoints are in: checkpoints/")
    print("- restoration_final.pth")
    print("- colorization_final.pth")

## 12. Summary & Next Steps

### What was trained:
1. **Restoration Model** - U-Net architecture to restore old/damaged photos
2. **Colorization Model** - Exemplar-based colorization network
3. **Similarity Subnet** - PyTorch VGG19-based semantic matching

### Trained checkpoints:
- `checkpoints/restoration_final.pth`
- `checkpoints/colorization_final.pth`

### To use locally:
1. Download `checkpoints.zip` from Colab
2. Extract to your local project directory
3. Run `python app.py` to start the Flask server
4. Open `index.html` in a browser
5. Upload old photo + reference → Get colorized result!

### Training time estimate:
- Restoration: ~3-4 hours (50 epochs)
- Colorization: ~2-3 hours (50 epochs)
- **Total: ~6-7 hours on Colab GPU**